In [1]:
from tradepy.data.loader import load_future
from tradepy.config.config import load_config, load_symbols, load_feature_config, load_exclude_config
from tradepy.data.cleaner import add_trading_date_by_gap
from tradepy.data.resampler import resample_ohlcv, daily_ohlcv_cummulative
from tradepy.features.generate import generate_features
from tradepy.features.quality_check import data_quality_report
from tradepy.supervised.load import prepare_all
from tradepy.paths import EXCLUDE_CONFIG, SYSTEM_CONFIG
from tradepy.supervised.target import target_triple_barrier_interday
from tradepy.supervised.train import filter_features

In [2]:
symbols = load_symbols()
cfg = load_config()

In [3]:
symbols

['AD',
 'BP',
 'CL',
 'EC',
 'ES',
 'GC',
 'MFXI',
 'NG',
 'NQ',
 'YM',
 'ZB',
 'ZN',
 'ZS']

In [4]:
ASSET = symbols[5]
MINUTES             = cfg['sampling_minutes']        # 240
RETURN_HORIZON_MIN  = cfg['return_horizon_min']      # 2880

In [5]:
df_final, spec = prepare_all(ASSET, MINUTES, RETURN_HORIZON_MIN, json_config_path="../Data/features_config.json")


In [6]:
spec

{'name': 'Gold',
 'symbol': 'GC',
 'ib_symbol': 'GC',
 'exchange': 'COMEX',
 'currency': 'USD',
 'approx_total_fee_per_side': 2.5,
 'min_slippage_ticks': 1,
 'max_spread_ticks': 4,
 'fallback_margin': 45000,
 'conId': 706903673,
 'multiplier': 100.0,
 'tick_size': 0.1,
 'tick_value': 10.0,
 'trading_hours': '20260425:CLOSED;20260426:1800-20260427:1700;20260427:1800-20260428:1330',
 'liquid_hours': '20260425:CLOSED;20260426:CLOSED;20260427:0930-20260427:1700;20260428:0930-20260428:1330',
 'min_profit_ticks': 2.5,
 'initial_margin': 44136.61,
 'maint_margin': 38379.66,
 'fetched_at': '2026-04-25T14:19:21.740096+00:00'}

In [7]:
exclude = load_exclude_config(MINUTES)

In [8]:
exclude

{'exclude_cols_triple': ['datetime',
  'dtyyyymmdd',
  'trading_date',
  'ticker',
  'per',
  'openint',
  'open',
  'high',
  'low',
  'close',
  'volume',
  'target_bin',
  'target_logret_240',
  'target_ret_240',
  'target_ticks_240'],
 'exclude_more': ['open_day',
  'bb_std',
  'true_range',
  'volume_cum',
  'vpt',
  'obv',
  'ad',
  'hour',
  'minute',
  'dayofweek',
  'day',
  'month',
  'swing_low',
  'swing_low_price',
  'swing_low_strength',
  'swing_high',
  'swing_high_price',
  'swing_high_strength']}

In [9]:
feature_cols_triple = [c for c in df_final.columns if c not in exclude['exclude_cols_triple']]

In [10]:
df_final_triple_real = target_triple_barrier_interday(df_final, spec, return_horizon_days=int(RETURN_HORIZON_MIN/60/24), profit_factor=1.8, tp_atr_mult=1.4, sl_atr_mult=1.2)
df_final_triple_real["target_class"] = df_final_triple_real[f"target_tb_{int(RETURN_HORIZON_MIN/60/24)}d"]

In [12]:
features_finales_test, params = filter_features(df_final_triple_real, feature_cols_triple, 'target_class')

Eliminando 59 features redundantes por correlación
Configuración sugerida:
{'lookback': 30, 'train_size': 3000, 'test_size': 400, 'gap': 12, '_candles_per_day': 6, '_horizon_bars': 12}
Features seleccionadas (60):
['volume_cum' 'signal' 'histogram' 'adx' 'rsi' 'slowk' 'cmo' 'williams_r'
 'cci' 'roc_10' 'roc_5' 'bb_std' 'hv' 'obv' 'obv_roc_5' 'obv_roc_10'
 'obv_z' 'ad_roc_5' 'ad_roc_10' 'ad_z' 'ad_rel' 'volume_range' 'mfi' 'cmf'
 'efi' 'vwap' 'avg_volume' 'relative_volume' 'volume_z' 'volume_delta'
 'pct_above_ma_20' 'pct_above_ma_50' 'slope_10' 'dist_low_cum_ticks'
 'dist_open_day_ticks' 'vol_weight_intraday' 'hour_cos' 'is_london'
 'return_horizon_2880m' 'return_lag_1' 'return_lag_5' 'return_lag_10'
 'return_lag_20' 'return_std_5' 'return_std_10' 'return_z_10'
 'rolling_skew_10' 'rolling_kurt_10' 'rolling_skew_20' 'rolling_kurt_20'
 'dist_to_last_swing_high' 'dist_to_last_swing_low' 'swing_range'
 'swing_range_pct' 'close_in_swing_range' 'atr_norm' 'atr_z'
 'dist_to_nearest_fib' 'swin